# Full primary CODI vs KaVa evaluation

This notebook performs **inference only** for the completed seed-0 CODI and KaVa checkpoints. It does not train either model. It replaces incomplete 200-example root prediction files with full-dataset predictions, while first preserving the current files under `capped_backup_before_full/`.

The notebook is restart-safe: on a later **Run all**, a method whose full prediction counts already pass validation is skipped. The paired report is generated only after both methods contain 1,319 GSM8K, 1,319 GSM-Hard, 300 SVAMP, and 180 MultiArith predictions.

In [ ]:
# Reproducible repository and Drive settings.
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "d917bef2cf396fe3b0453e6f86648f1a3948f528"
REPO_DIR = "/content/latent-reasoning"
DRIVE_ROOT = "/content/drive/MyDrive/CODI_KAVA"
LOCAL_ROOT = "/content/codikava_runtime"
FINAL_STEP = 96405
METHODS_TO_RUN = ("codi", "kava")
FORCE_RERUN = False  # Keep False: completed full evaluations will be skipped.
BOOTSTRAP_SAMPLES = 10000
EXPECTED_COUNTS = {
    "gsm8k": 1319,
    "gsm_hard": 1319,
    "svamp": 300,
    "multiarith": 180,
}
assert set(METHODS_TO_RUN).issubset({"codi", "kava"})

## 1. Mount Drive and prepare the pinned repository

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import datetime
import json
import os
import pathlib
import shutil
import subprocess
import sys

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
pathlib.Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", RUN_COMMIT], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")],
    check=True,
)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert commit == RUN_COMMIT, (commit, RUN_COMMIT)
print("Checked out:", commit)

## 2. Verify the GPU and final checkpoints

In [ ]:
import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU"
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

from scripts.colab_runner import validate_torch_checkpoint_archive

for method in METHODS_TO_RUN:
    checkpoint = (
        pathlib.Path(DRIVE_ROOT)
        / "outputs"
        / method
        / "checkpoints"
        / f"step_{FINAL_STEP:08d}.pt"
    )
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Missing final {method} checkpoint: {checkpoint}")
    validate_torch_checkpoint_archive(checkpoint)
    print(method.upper(), f"{checkpoint.stat().st_size / 2**30:.2f} GiB", checkpoint)

## 3. Define restart-safe evaluation helpers

In [ ]:
def evaluation_dir(method):
    return (
        pathlib.Path(DRIVE_ROOT)
        / "outputs"
        / method
        / "eval"
        / f"step_{FINAL_STEP:08d}"
    )


def jsonl_count(path):
    if not path.is_file():
        return 0
    with path.open(encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip())


def evaluation_counts(method):
    root = evaluation_dir(method)
    return {name: jsonl_count(root / f"{name}.jsonl") for name in EXPECTED_COUNTS}


def evaluation_is_full(method):
    return evaluation_counts(method) == EXPECTED_COUNTS


def print_counts(method):
    counts = evaluation_counts(method)
    print(f"\n{method.upper()}")
    for dataset, expected in EXPECTED_COUNTS.items():
        print(f"{dataset:12s}: {counts[dataset]}/{expected}")
    return counts


def backup_incomplete_evaluation(method):
    counts = evaluation_counts(method)
    if not any(counts.values()) or counts == EXPECTED_COUNTS:
        return None
    source = evaluation_dir(method)
    backup = source / "capped_backup_before_full"
    backup.mkdir(parents=True, exist_ok=True)
    for filename in [*(f"{name}.jsonl" for name in EXPECTED_COUNTS), "summary.json"]:
        src = source / filename
        dst = backup / filename
        if src.is_file() and not dst.exists():
            shutil.copy2(src, dst)
    audit = backup / "counts_before_full.json"
    if not audit.exists():
        audit.write_text(json.dumps(counts, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(f"Preserved existing {method} predictions at {backup}")
    return backup


def run_persisted(cmd, log_name):
    logs = pathlib.Path(DRIVE_ROOT) / "logs"
    logs.mkdir(parents=True, exist_ok=True)
    log_path = logs / log_name
    print("Starting:", " ".join(map(str, cmd)), flush=True)
    print("Persistent log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat()
        log.write(f"\n=== {timestamp} {' '.join(map(str, cmd))} ===\n")
        process = subprocess.Popen(
            cmd,
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        return_code = process.wait()
        log.flush()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    return return_code


def run_full_evaluation(method):
    before = print_counts(method)
    if before == EXPECTED_COUNTS and not FORCE_RERUN:
        print(f"Skipping {method}: full evaluation is already complete.")
        return
    backup_incomplete_evaluation(method)
    cmd = [
        sys.executable,
        "-u",
        "scripts/colab_ablation_runner.py",
        "--method", method,
        "--drive-root", DRIVE_ROOT,
        "--local-root", LOCAL_ROOT,
        "--mode", "baseline",
        "--limit", "0",
    ]
    run_persisted(cmd, f"{method}_full_eval.log")
    after = print_counts(method)
    if after != EXPECTED_COUNTS:
        raise RuntimeError(f"{method} evaluation is still incomplete: {after}")
    print(f"{method.upper()} full evaluation verified and persisted to Drive.")

## 4. Inspect current prediction counts

Counts of 200 for GSM8K, GSM-Hard, or SVAMP indicate the old capped evaluation.

In [ ]:
for method in METHODS_TO_RUN:
    print_counts(method)

## 5. Run or resume the full CODI evaluation

This is a separate cell so that a completed CODI result remains easy to verify before KaVa starts. On a restarted runtime, this cell skips CODI if its Drive counts are already complete.

In [ ]:
if "codi" in METHODS_TO_RUN:
    run_full_evaluation("codi")
else:
    print("CODI disabled in METHODS_TO_RUN")

## 6. Run or resume the full KaVa evaluation

In [ ]:
if "kava" in METHODS_TO_RUN:
    run_full_evaluation("kava")
else:
    print("KaVa disabled in METHODS_TO_RUN")

## 7. Validate both methods and generate the full paired report

This cell deliberately refuses to compare a full run with a partial run.

In [ ]:
for method in ("codi", "kava"):
    counts = print_counts(method)
    if counts != EXPECTED_COUNTS:
        raise RuntimeError(f"Cannot build full report: {method} counts are {counts}")

from IPython.display import Markdown, display

reports = pathlib.Path(DRIVE_ROOT) / "reports"
reports.mkdir(parents=True, exist_ok=True)
full_output = reports / "codi_vs_kava_full.json"
cmd = [
    sys.executable,
    "scripts/analyze_phase2.py",
    "--run", f"codi={evaluation_dir('codi')}",
    "--run", f"kava={evaluation_dir('kava')}",
    "--output", str(full_output),
    "--bootstrap-samples", str(BOOTSTRAP_SAMPLES),
]
subprocess.run(cmd, cwd=REPO_DIR, check=True)
markdown_output = full_output.with_suffix(".md")
display(Markdown(markdown_output.read_text()))
print("Full JSON report:", full_output)
print("Full Markdown report:", markdown_output)

## 8. Final durable-artifact summary

In [ ]:
print("Evaluation complete. Durable artifacts:")
for method in ("codi", "kava"):
    root = evaluation_dir(method)
    print(f"  {method}: {root}")
    print(f"  log: {pathlib.Path(DRIVE_ROOT) / 'logs' / f'{method}_full_eval.log'}")
print(f"  report: {pathlib.Path(DRIVE_ROOT) / 'reports' / 'codi_vs_kava_full.json'}")